<a href="https://colab.research.google.com/github/Krishnan-Raghavan/Packt/blob/main/StableDiffusionChapter12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
!pip install diffusers
!pip install transformers scipy ftfy accelerate ipywidgets

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5"
    , torch_dtype=torch.float16)
pipe.to("cuda")
prompt = "a photo of an astronaut riding a horse on mars,blazing fast, wind and sand moving back"
image = pipe(
    prompt, num_inference_steps=30
).images[0]
image

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
pipeline = StableDiffusionPipeline.from_pretrained(
    "stablediffusionapi/deliberate-v2",
    torch_dtype = torch.float16,
    safety_checker = None
).to("cuda:0")
image = pipeline(
    prompt = "A photo with half cat and half dog"
    , generator = torch.Generator("cuda:0").manual_seed(3)
).images[0]
image

In [ ]:
!pip install compel

In [ ]:
import compel

In [ ]:
from compel import Compel
compel = Compel(
    tokenizer       = pipeline.tokenizer
    , text_encoder  = pipeline.text_encoder
)

In [ ]:
prompt = '("A photo of cat", "A photo of dog").blend(0.5, 0.5)'
prompt_embeds = compel(prompt)

In [ ]:
image = pipeline(
    prompt_embeds   = prompt_embeds
    , generator     = torch.Generator("cuda:0").manual_seed(1)
).images[0]
image

In [ ]:
prompt = '("A photo of cat", "A photo of dog").blend(0.7, 0.3)'
prompt_embeds = compel(prompt)
image = pipeline(
    prompt_embeds   = prompt_embeds
    , generator     = torch.Generator("cuda:0").manual_seed(1)
).images[0]
image

In [ ]:
prompt = '("A photo of cat", "A photo of dog").blend(0.7, 0.3)'
prompt_embeds = compel(prompt)
image = pipeline(
    prompt_embeds   = prompt_embeds
    , generator     = torch.Generator("cuda:0").manual_seed(1)
).images[0]
image

In [ ]:
prompt = '[A photo of cat:A photo of dog:0.5]'
prompt_embeds = compel(prompt)
image = pipeline(
    prompt_embeds   = prompt_embeds
    , generator     = torch.Generator("cuda:0").manual_seed(1)
).images[0]
image

In [ ]:
prompt = '[A photo of cat|A photo of dog]'
prompt_embeds = compel(prompt)
image = pipeline(
    prompt_embeds   = prompt_embeds
    , generator     = torch.Generator("cuda:0").manual_seed(1)
).images[0]
image

In [ ]:
prompt = '[A photo of cat:A photo of dog:0.5]'
prompt_embeds = compel(prompt)
image = pipeline(
    prompt_embeds   = prompt_embeds
    , generator     = torch.Generator("cuda:0").manual_seed(1)
).images[0]
image

In [ ]:
!pip install -U lark

In [ ]:
import lark
schedule_parser = lark.Lark(r"""
!start: (prompt | /[][():]/+)*
prompt: (emphasized | scheduled | alternate | plain | WHITESPACE)*
!emphasized: "(" prompt ")"
        | "(" prompt ":" prompt ")"
        | "[" prompt "]"
scheduled: "[" [prompt ":"] prompt ":" [WHITESPACE] NUMBER "]"
alternate: "[" prompt ("|" prompt)+ "]"
WHITESPACE: /\s+/
plain: /([^\\\[\]():|]|\\.)+/
%import common.SIGNED_NUMBER -> NUMBER
""")

In [ ]:
def get_learned_conditioning_prompt_schedules(prompts, steps):
    def collect_steps(steps, tree):
        l = [steps]
        class CollectSteps(lark.Visitor):
            def scheduled(self, tree):
                tree.children[-1] = float(tree.children[-1])
                if tree.children[-1] < 1:
                    tree.children[-1] *= steps
                tree.children[-1] = min(steps, int(tree.children[-1]))
                l.append(tree.children[-1])
            def alternate(self, tree):
                l.extend(range(1, steps+1))
        CollectSteps().visit(tree)
        return sorted(set(l))

    def at_step(step, tree):
        class AtStep(lark.Transformer):
            def scheduled(self, args):
                before, after, _, when = args
                yield before or () if step <= when else after
            def alternate(self, args):
                yield next(args[(step - 1)%len(args)])
            def start(self, args):
                def flatten(x):
                    if type(x) == str:
                        yield x
                    else:
                        for gen in x:
                            yield from flatten(gen)
                return ''.join(flatten(args))
            def plain(self, args):
                yield args[0].value
            def __default__(self, data, children, meta):
                for child in children:
                    yield child
        return AtStep().transform(tree)

    def get_schedule(prompt):
        try:
            tree = schedule_parser.parse(prompt)
        except lark.exceptions.LarkError as e:
            if 0:
                import traceback
                traceback.print_exc()
            return [[steps, prompt]]
        return [[t, at_step(t, tree)] for t in collect_steps(steps, tree)]

    promptdict = {prompt: get_schedule(prompt) for prompt in set(prompts)}
    return [promptdict[prompt] for prompt in prompts]

In [ ]:
steps = 10
g = lambda p: get_learned_conditioning_prompt_schedules([p], steps)[0]

In [ ]:
g("cat")

In [ ]:
[[10, 'cat']]

In [ ]:
g('[cat:dog:0.5]')

In [ ]:
[[5, 'cat'], [10, 'dog']]

In [ ]:
g('[cat|dog]')

In [ ]:
def parse_scheduled_prompts(text, steps=10):
    text = text.strip()
    parse_result = None
    try:
        parse_result = get_learned_conditioning_prompt_schedules(
            [text],
            steps = steps
        )[0]
    except Exception as e:
        print(e)

    if len(parse_result) == 1:
        return parse_result

    prompts_list = []

    for i in range(steps):
        current_prompt_step, current_prompt_content = parse_result[0][0],parse_result[0][1]
        step = i + 1
        if step < current_prompt_step:
            prompts_list.append(current_prompt_content)
            continue

        if step == current_prompt_step:
            prompts_list.append(current_prompt_content)
            parse_result.pop(0)

    return prompts_list

In [ ]:
prompt_list = parse_scheduled_prompts("[cat:dog:0.5]")
prompt_list

In [ ]:
prompt_embeds = self._encode_prompt(
    prompt,
    device,
    num_images_per_prompt,
    do_classifier_free_guidance,
    negative_prompt,
    negative_prompt_embeds=negative_prompt_embeds,
)

In [ ]:
from typing import List, Callable, Dict, Any
from torch import Generator,FloatTensor
from diffusers.pipelines.stable_diffusion import StableDiffusionPipelineOutput
from diffusers import StableDiffusionPipeline,EulerDiscreteScheduler

class StableDiffusionPipeline_EXT(StableDiffusionPipeline):
    @torch.no_grad()
    def scheduler_call(
        self
        , prompt: str | List[str] = None
        , height: int | None = 512
        , width: int | None = 512
        , num_inference_steps: int = 50
        , guidance_scale: float = 7.5
        , negative_prompt: str | List[str] | None = None
        , num_images_per_prompt: int | None = 1
        , eta: float = 0
        , generator: Generator | List[Generator] | None = None
        , latents: FloatTensor | None = None
        , prompt_embeds: FloatTensor | None = None
        , negative_prompt_embeds: FloatTensor | None = None
        , output_type: str | None = "pil"
        , callback: Callable[[int, int, FloatTensor], None] | None = None
        , callback_steps: int = 1
        , cross_attention_kwargs: Dict[str, Any] | None = None
    ):
        ...
        extra_step_kwargs = self.prepare_extra_step_kwargs(generator, eta)
        num_warmup_steps = len(timesteps) - num_inference_steps * self.scheduler.order
        with self.progress_bar(total=num_inference_steps) as progress_bar:
            for i, t in enumerate(timesteps):
                # AZ code to enable Prompt Scheduling, will only function when
                # when there is a prompt_embeds_l provided.
                prompt_embeds_l_len = len(embedding_list)
                if prompt_embeds_l_len > 0:
                    # ensure no None prompt will be used
                    pe_index = (i)%prompt_embeds_l_len
                    prompt_embeds = embedding_list[pe_index]

                # expand the latents if we are doing classifier free guidance
                latent_model_input = torch.cat([latents] * 2) if do_classifier_free_guidance else latents
                latent_model_input = self.scheduler.scale_model_input(latent_model_input, t)

                # predict the noise residual
                noise_pred = self.unet(
                    latent_model_input,
                    t,
                    encoder_hidden_states=prompt_embeds,
                    cross_attention_kwargs=cross_attention_kwargs,
                ).sample

                # perform guidance
                if do_classifier_free_guidance:
                    noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
                    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

                # compute the previous noisy sample x_t -> x_t-1
                latents = self.scheduler.step(noise_pred, t, latents, **extra_step_kwargs).prev_sample

                # call the callback, if provided
                if i == len(timesteps) - 1 or ((i + 1) > num_warmup_steps and (i + 1) % self.scheduler.order == 0):
                    progress_bar.update()
                    if callback is not None and i % callback_steps == 0:
                        callback(i, t, latents)

        if output_type == "latent":
            image = latents
        elif output_type == "pil":
            # 8. Post-processing
            image = self.decode_latents(latents)
            image = self.numpy_to_pil(image)
        else:
            # 8. Post-processing
            image = self.decode_latents(latents)

        if hasattr(self, "final_offload_hook") and self.final_offload_hook is not None:
            self.final_offload_hook.offload()

        return StableDiffusionPipelineOutput(images=image)

In [ ]:
if self.scheduler._class_name == "PNDMScheduler":
    self.scheduler = EulerDiscreteScheduler.from_config(
        self.scheduler.config
    )

In [ ]:
device = self._execution_device
do_classifier_free_guidance = guidance_scale > 1.0

In [ ]:
prompt_list = parse_scheduled_prompts(prompt)

In [ ]:
embedding_list = []
if len(prompt_list) == 1:
    prompt_embeds = self._encode_prompt(
        prompt,
        device,
        num_images_per_prompt,
        do_classifier_free_guidance,
        negative_prompt,
        negative_prompt_embeds=negative_prompt_embeds,
    )
else:
    for prompt in prompt_list:
        prompt_embeds = self._encode_prompt(
            prompt,
            device,
            num_images_per_prompt,
            do_classifier_free_guidance,
            negative_prompt,
            negative_prompt_embeds=negative_prompt_embeds,
        )
        embedding_list.append(prompt_embeds)

In [ ]:
self.scheduler.set_timesteps(num_inference_steps, device=device)
timesteps = self.scheduler.timesteps

In [ ]:
num_channels_latents = self.unet.in_channels
batch_size = 1
latents = self.prepare_latents(
    batch_size * num_images_per_prompt,
    num_channels_latents,
    height,
    width,
    prompt_embeds.dtype,
    device,
    generator,
    latents,
)

In [ ]:
num_warmup_steps = len(timesteps) - num_inference_steps * self.scheduler.order
with self.progress_bar(total=num_inference_steps) as progress_bar:
    for i, t in enumerate(timesteps):
        # custom code to enable Prompt Scheduling, will only function when
        # when there is a prompt_embeds_l provided.
        prompt_embeds_l_len = len(embedding_list)
        if prompt_embeds_l_len > 0:
            # ensure no None prompt will be used
            pe_index = (i)%prompt_embeds_l_len
            prompt_embeds = embedding_list[pe_index]

        # expand the latents if we are doing classifier free guidance
        latent_model_input = torch.cat([latents] * 2) if do_classifier_free_guidance else latents
        latent_model_input = self.scheduler.scale_model_input(latent_model_input, t)

        # predict the noise residual
        noise_pred = self.unet(
            latent_model_input,
            t,
            encoder_hidden_states=prompt_embeds,
            cross_attention_kwargs=cross_attention_kwargs,
        ).sample

        # perform guidance
        if do_classifier_free_guidance:
            noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        # compute the previous noisy sample x_t -> x_t-1
        latents = self.scheduler.step(noise_pred, t, latents).prev_sample

        # call the callback, if provided
        if i == len(timesteps) - 1 or ((i + 1) > num_warmup_steps and (i + 1) % self.scheduler.order == 0):
            progress_bar.update()
            if callback is not None and i % callback_steps == 0:
                callback(i, t, latents)

In [ ]:
image = self.decode_latents(latents)
image = self.numpy_to_pil(image)
return StableDiffusionPipelineOutput(images=image, nsfw_content_detected=None)

In [ ]:
pipeline = StableDiffusionPipeline_EXT.from_pretrained(
    "stablediffusionapi/deliberate-v2",
    torch_dtype = torch.float16,
    safety_checker = None
).to("cuda:0")
prompt = "high quality, 4k, details, A realistic photo of cute [cat:dog:0.6]"
neg_prompt = "paint, oil paint, animation, blur, low quality, bad glasses"
image = pipeline.scheduler_call(
    prompt = prompt
    , negative_prompt = neg_prompt
    , generator = torch.Generator("cuda").manual_seed(1)
).images[0]
image

In [ ]:
prompt = "high quality, 4k, details, A realistic photo of white [cat|dog]"
neg_prompt = "paint, oil paint, animation, blur, low quality, bad glasses"
image = pipeline.scheduler_call(
    prompt = prompt
    , negative_prompt = neg_prompt
    , generator = torch.Generator("cuda").manual_seed(3)
).images[0]
image